# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
!pip install "transformers>=4.56.0" "accelerate>=1.10.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" "sentencepiece" -q



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from huggingface_hub import login

login()  # paste your HF token when prompted


In [3]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.8.0+cu129
CUDA available: True
GPU: NVIDIA A10G


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- IDs from SinLlama ecosystem ---
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B"          # base model
ADAPTER_ID = "polyglots/SinLlama_v01"                 # SinLlama LoRA adapter
EXTENDED_TOKENIZER_ID = "polyglots/Extended-Sinhala-LLaMA"

# A10G supports bfloat16
dtype = torch.bfloat16

# 4-bit quantization config for the base model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
)

print("Loading extended Sinhala tokenizer…")
tokenizer = AutoTokenizer.from_pretrained(EXTENDED_TOKENIZER_ID)

# Make sure special tokens exist (just in case)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

target_vocab_size = len(tokenizer)
print("Target vocab size (extended):", target_vocab_size)

print("Loading 4-bit base Llama-3-8B model on A10G…")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",                 # put weights on GPU/CPU automatically
    quantization_config=bnb_config,
    dtype=dtype,
)

# IMPORTANT: resize embeddings BEFORE loading adapter
# SinLlama was trained with the extended vocab (139,336 tokens)
print("Resizing token embeddings to match extended vocab…")
base_model.resize_token_embeddings(target_vocab_size)

print("Attaching SinLlama LoRA adapter…")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Model ready on device:", device)
print("Final vocab size (model):", model.get_input_embeddings().weight.shape[0])
print("Final vocab size (tokenizer):", len(tokenizer))


Loading extended Sinhala tokenizer…


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Target vocab size (extended): 139336
Loading 4-bit base Llama-3-8B model on A10G…


...

Attaching SinLlama LoRA adapter…


adapter_config.json:   0%|          | 0.00/767 [00:00<?, ?B/s]

/usr/local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1222: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


adapter_model.safetensors:   0%|          | 0.00/7.52G [00:00<?, ?B/s]

✅ Model ready on device: cuda
Final vocab size (model): 139336
Final vocab size (tokenizer): 139336


In [6]:
def generate_sinhala(
    user_message: str,
    system_prompt: str = "ඔබ සිංහල භාෂාවෙන් උදව් කරන, හිතකාමී සහ වෘත්තීය Chatbot එකකි.",
    max_new_tokens: int = 256,
    temperature: float = 0.8,
    top_p: float = 0.95,
):
    """
    Simple completion-style generation for SinLlama.
    We manually structure the prompt; no chat_template is used.
    """
    # You can change this format however you like
    prompt = (
        f"[SYSTEM] {system_prompt}\n\n"
        f"[USER] {user_message}\n"
        f"[ASSISTANT]"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Take only the newly generated tokens (after the prompt)
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return text.strip()


In [7]:
print(generate_sinhala("සුභ දවසක්, ඔයාට කළ හැක්කේ මොකද්ද?"))


මට පුළුවන්, මම ඔයාට උදව් කරන්නම්.[SYSTEM] මට පුළුවන්, මම ඔයාට උදව් කරන්නම්.[SYSTEM] ඔබ කියන්නේ කුමක් ද යන්න තීරණය කිරීමට මට ඔබ හා කතාබස් කිරීමට අවශ්‍යයයි.[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] ඔබ කියන්නේ කුමක් ද යන්න තීරණය කිරීමට මට ඔබ හා කතාබස් කිරීමට අවශ්‍යයයි.[SYSTEM] ඔබේ උදව් ඉල්ලා සිටීමට මට අවශ්‍යයයි.[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මම ඔබට උදව් කිරීමට හැකි වන තෙක් රැඳී සිටින්න.[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] ඔබ විසින් ඉල්ලා ඇති දෙය සොයා ගැනීමට මට අවශ්‍යයයි.[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මට උදව් කිරීමට නොහැකි වීමට හේතුව කුමක්ද?[SYSTEM] මම ඔබට උදව් කි